# Uncertainty Characterization and Modeling

Demand is uncertain, but the available information is currently very limited. Three expert suggests that the estimation error be ±10%, ±20% or ±30% around the nominal demand. No further statistical information or worst-case information is provided. Your design should remain feasible under the uncertainty information provided. Show to the three experts how their estimate affect the solution.

Nominal demand: $u^{\text{nom}} = [22, 10, 25, 10]^\top$

## 1. Uncertainty Characterization

We treat this uncertainty as **epistemic**. The reason is straightforward: we do not have a probability distribution, historical data, or any statistical description of how demand varies. What we have are three experts giving different opinions on how wrong the nominal estimate could be. That is a lack of knowledge, not inherent variability; and it would be reducible if more information were available.

Given this, we model the uncertainty with a **box uncertainty set** and solve a **robust optimization problem**. The box set is the natural fit here: each demand $u_i$ can deviate from its nominal value by at most $\pm \epsilon \cdot u_i^{\text{nom}}$, with no constraint on how many components deviate at once. We solve the problem once per expert estimate ($\epsilon \in \{10\%, 20\%, 30\%\}$) to see how the assumed deviation size affects the design.


## 2. Uncertainty Quantification


For each expert $k$, we define a box uncertainty set centered at the nominal demand:

$$\mathcal{U}_k = \left\{ u \in \mathbb{R}^4 : \left| u_i - u_i^{\text{nom}} \right| \leq \epsilon_k \cdot u_i^{\text{nom}}, \quad \forall i \right\}, \qquad \epsilon_k \in \{0.10,\ 0.20,\ 0.30\}$$

Each component of $u$ can deviate independently from its nominal value by up to $\epsilon_k$ in relative terms. No additional constraint is placed on how many components deviate at once.

This changes the optimization problem in a concrete way: instead of designing for the nominal demand, we now need the capacity constraints to hold for every $u \in \mathcal{U}_k$. The design becomes more conservative as $\epsilon_k$ grows, and so does the cost. Since the box set preserves LP tractability, the robust problem has the same structure as the original.

## 3. Decision-making Problem

In [1]:
# Imports
from sys import path
path.insert(0, '../src')

from scipy.optimize import linprog
import numpy as np
from problem import c, u_nom, A_ineq, B, bounds

In [ ]:
epsilons = [0.10, 0.20, 0.30]

for eps in epsilons:
    # same LP structure as the deterministic case; only b changes
    b_worst = B @ u_nom + eps * np.abs(B) @ u_nom
    result = linprog(c, A_ub=-A_ineq, b_ub=-b_worst, bounds=bounds, method='highs')
    a_star = result.x
    print(f"epsilon = {eps:.0%}")
    print(f"  x* = {a_star[:3].round(2)}")
    print(f"  y* = {a_star[3:].round(2)}")
    print(f"  cost = {result.fun:.2f}\n")

epsilon = 10%
  x* = [10.  53.7 10. ]
  y* = [ 1.  31.2  1. ]
  cost = 385.90

epsilon = 20%
  x* = [10.  60.4 10. ]
  y* = [ 2.  40.4  2. ]
  cost = 452.80

epsilon = 30%
  x* = [10.  67.1 10. ]
  y* = [ 3.  49.6  3. ]
  cost = 519.70

